# Feature Engineering 
Build one row per client-renewal-cycle with features calculated using only information available on or before the prediction month. 

I engineered leakage-safe renewal-risk features using only data available up to three months before contract end. Features captured recent engagement, support tickets, and satisfaction levels (nps), as well as changes versus the prior quarter to represent early deterioration signals before renewal.

## Important Rule 
Every feature must only use data up to prediction_month because we are predicting 3 months before renewal

In [0]:
from pyspark.sql import functions as F

In [0]:
client_monthly_model_base = spark.table("client_monthly_model_base")
prediction_snapshot = spark.table("prediction_snapshot_valid")

In [0]:
print("prediction_snapshot rows:", prediction_snapshot.count())

In [0]:
# For each client snapshot, I'll have its historical monthly records leading up to prediction month. One prediction row expands into multiple monthly rows.
feature_base = (
    prediction_snapshot.alias("p")
    .join(client_monthly_model_base.alias("m"), 
          (F.col("p.client_id") == F.col("m.client_id")) &
          (F.col("m.month") <= F.col("p.prediction_month")),
    how = "left"
)
.select(
    F.col("p.client_id"),
        F.col("p.renewal_cycle_id"),
        F.col("p.prediction_month"),
        F.col("p.contract_end_date"),
        F.col("p.renewed_flag"),
        F.col("p.non_renewal_flag"),
        F.col("p.industry"),
        F.col("p.region"),
        F.col("p.company_size"),
        F.col("p.eligible_employees"),
        F.col("p.annual_contract_value"),
        F.col("p.contract_term_months"),
        F.col("m.month"),
        F.col("m.utilization_rate"),
        F.col("m.sessions"),
        F.col("m.active_members"),
        F.col("m.webinar_attendance"),
        F.col("m.employer_comms_sent"),
        F.col("m.ticket_count"),
        F.col("m.high_severity_tickets"),
        F.col("m.nps")
    )
)

display(feature_base)

In [0]:
# Adding a helper column for how far each month is from the prediction month 
feature_base = (
    feature_base 
    .withColumn(
        "months_before_prediction",
        F.floor(F.months_between(F.col("prediction_month"), F.col("month")))
    )
)

display(feature_base)

In [0]:
# Aggregate last 3 months of features
features_last_3m = (
    feature_base
    .filter(F.col("months_before_prediction").between(0,2))
    .groupBy(
        "client_id",
        "renewal_cycle_id",
        "prediction_month",
        "contract_end_date",
        "renewed_flag",
        "non_renewal_flag",
        "industry",
        "region",
        "company_size",
        "eligible_employees",
        "annual_contract_value",
        "contract_term_months"
    )
    .agg(
        F.avg("utilization_rate").alias("utilization_avg_last_3m"),
        F.avg("sessions").alias("sessions_avg_last_3m"),
        F.avg("active_members").alias("active_members_avg_last_3m"),
        F.avg("webinar_attendance").alias("webinar_attendance_avg_last_3m"),
        F.avg("employer_comms_sent").alias("employer_comms_avg_last_3m"),
        F.avg("ticket_count").alias("ticket_count_avg_last_3m"),
        F.avg("high_severity_tickets").alias("high_severity_tickets_avg_last_3m"),
        F.avg("nps").alias("nps_avg_last_3m")
    )
)

display(features_last_3m)

In [0]:
# Aggregate prior 3 months of features to measure deterioration to answer the question: Is engagement getting worse?
features_prior_3m = (
    feature_base
    .filter(F.col("months_before_prediction").between(3, 5))
    .groupBy(
        "client_id",
        "renewal_cycle_id"
    )
    .agg(
        F.avg("utilization_rate").alias("utilization_avg_prior_3m"),
        F.avg("sessions").alias("sessions_avg_prior_3m"),
        F.avg("active_members").alias("active_members_avg_prior_3m"),
        F.avg("webinar_attendance").alias("webinar_attendance_avg_prior_3m"),
        F.avg("employer_comms_sent").alias("employer_comms_avg_prior_3m"),
        F.avg("ticket_count").alias("ticket_count_avg_prior_3m"),
        F.avg("high_severity_tickets").alias("high_severity_tickets_avg_prior_3m"),
        F.avg("nps").alias("nps_avg_prior_3m")
    )
)

display(features_prior_3m)

In [0]:
# Join both windows and compute absolute change (instead of percetage change) for stability (percentage change explodes when the denominator is small, this can introduce extreme values), interpretability and consistency across metrics 
# Interpretation: utilization_change_3m_vs_prior_3m < 0 then utilization is worse 
# Interpretation: ticket_count_change_3m_vs_prior_3m > 0 then support tickets increase, getting worse 
# Interpretation: nps_change_3m_vs_prior_3m < 0 then NPS is worse, satisfaction declined
renewal_model_dataset = (
features_last_3m.alias("l")
.join(
    features_prior_3m.alias("p"),
    on=["client_id", "renewal_cycle_id"],
    how="left"
)
.withColumn(
    "utilization_change_3m_vs_prior_3m",
    F.col("utilization_avg_last_3m") - F.col("utilization_avg_prior_3m")
)
.withColumn(
    "sessions_change_3m_vs_prior_3m",
    F.col("sessions_avg_last_3m") - F.col("sessions_avg_prior_3m")
)
.withColumn(
    "active_members_change_3m_vs_prior_3m",
    F.col("active_members_avg_last_3m") - F.col("active_members_avg_prior_3m")
)
.withColumn(
    "webinar_attendance_change_3m_vs_prior_3m",
    F.col("webinar_attendance_avg_last_3m") - F.col("webinar_attendance_avg_prior_3m")
)
.withColumn(
    "employer_comms_change_3m_vs_prior_3m",
    F.col("employer_comms_avg_last_3m") - F.col("employer_comms_avg_prior_3m")
)
.withColumn(
    "ticket_count_change_3m_vs_prior_3m",
    F.col("ticket_count_avg_last_3m") - F.col("ticket_count_avg_prior_3m")
)
.withColumn(
    "high_severity_tickets_change_3m_vs_prior_3m",
    F.col("high_severity_tickets_avg_last_3m") - F.col("high_severity_tickets_avg_prior_3m")
)
.withColumn(
    "nps_change_3m_vs_prior_3m",
    F.col("nps_avg_last_3m") - F.col("nps_avg_prior_3m")
)
)

display(renewal_model_dataset)

In [0]:
# Add simple ratio features to normalize
renewal_model_dataset = (
    renewal_model_dataset
    .withColumn(
        "active_member_rate_last_3m",
        F.col("active_members_avg_last_3m") / F.col("eligible_employees")
    )
    .withColumn(
        "sessions_per_active_member_last_3m",
        F.col("sessions_avg_last_3m") / F.col("active_members_avg_last_3m")
    )
    .withColumn(
        "tickets_per_1000_eligible_last_3m",
        (F.col("ticket_count_avg_last_3m") / F.col("eligible_employees")) * 1000
    )
)

display(renewal_model_dataset)

## Sanity Checks

In [0]:
print("prediction_snapshot rows:", prediction_snapshot.count())
print("renewal_model_dataset rows:", renewal_model_dataset.count())

In [0]:
# Duplicate check 
display(
    renewal_model_dataset
    .groupBy("client_id", "renewal_cycle_id")
    .count()
    .filter(F.col("count") > 1)
)

In [0]:
# Nulls check 
display(
    renewal_model_dataset.select(
        [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in renewal_model_dataset.columns]
    )
)

In [0]:
display(
    renewal_model_dataset
    .groupBy("non_renewal_flag")
    .agg(
        F.avg("utilization_avg_last_3m").alias("avg_utilization"),
        F.avg("ticket_count_avg_last_3m").alias("avg_ticket_count"),
        F.avg("nps_avg_last_3m").alias("avg_nps"),
        F.avg("utilization_change_3m_vs_prior_3m").alias("avg_utilization_change"),
        F.avg("ticket_count_change_3m_vs_prior_3m").alias("avg_ticket_change"),
        F.avg("nps_change_3m_vs_prior_3m").alias("avg_nps_change")
    )
)

In [0]:
spark.sql("DROP TABLE IF EXISTS renewal_model_dataset")
renewal_model_dataset.write.saveAsTable("renewal_model_dataset  ")